# 01 · Bronze Ingest

Reads Parquet files from `/Volumes/asset_mgmt/bronze/uploads/`, stamps three metadata columns,
and **appends** to `/Volumes/asset_mgmt/bronze/prices/` partitioned by `ingest_date`.

| Widget | Default | Values |
|---|---|---|
| `source_type` | `batch_2` | `seed` · `batch_2` · `live` |
| `uploads_path` | `/Volumes/asset_mgmt/bronze/uploads/` | any Volume path |


In [ ]:
# ── Widgets (set before running; safe to re-run) ──────────────────────────
dbutils.widgets.removeAll()
dbutils.widgets.text("source_type",  "batch_2",                              "Source type (seed|batch_2|live)")
dbutils.widgets.text("uploads_path", "/Volumes/asset_mgmt/bronze/uploads/",  "Source folder")

In [ ]:
from pyspark.sql import functions as F

SOURCE_TYPE  = dbutils.widgets.get("source_type")
UPLOADS_PATH = dbutils.widgets.get("uploads_path")
PRICES_PATH  = "/Volumes/asset_mgmt/bronze/prices/"

print(f"source_type  : {SOURCE_TYPE}")
print(f"uploads_path : {UPLOADS_PATH}")
print(f"prices_path  : {PRICES_PATH}")

In [ ]:
# ── Guard: abort early if the uploads folder is empty ────────────────────
upload_files = dbutils.fs.ls(UPLOADS_PATH)
parquet_files = [f for f in upload_files if f.name.endswith(".parquet") or f.isDir()]

if not parquet_files:
    raise FileNotFoundError(
        f"No Parquet files found in {UPLOADS_PATH}. "
        "Upload your files before running this notebook."
    )

print(f"Found {len(parquet_files)} item(s) in uploads folder — proceeding.")

In [ ]:
# ── Read + stamp metadata ─────────────────────────────────────────────────
# Unity Catalog requires _metadata.file_path instead of input_file_name()
raw = (
    spark.read
        .option("recursiveFileLookup", "true")
        .parquet(UPLOADS_PATH)
        .withColumn("_source_file",      F.col("_metadata.file_path"))
        .withColumn("_ingest_timestamp", F.current_timestamp())
        .withColumn("_source_type",      F.lit(SOURCE_TYPE))
        .withColumn("ingest_date",        F.to_date(F.current_timestamp()))
)

row_count = raw.count()
print(f"Rows read    : {row_count:,}")
print(f"Tickers      : {raw.select('Ticker').distinct().count()}")
raw.printSchema()

In [ ]:
# ── Preview ───────────────────────────────────────────────────────────────
display(raw.limit(10))

In [ ]:
# ── Append-only write to bronze/prices, partitioned by ingest_date ────────
(
    raw.write
        .mode("append")
        .partitionBy("ingest_date")
        .parquet(PRICES_PATH)
)

print(f"Appended {row_count:,} rows → {PRICES_PATH}")

In [ ]:
# ── Verify: row counts by ingest_date and source_type ────────────────────
verify = spark.read.parquet(PRICES_PATH)

print(f"Total rows in bronze/prices: {verify.count():,}")
display(
    verify
        .groupBy("ingest_date", "_source_type")
        .agg(
            F.count("*").alias("rows"),
            F.countDistinct("Ticker").alias("tickers"),
            F.min("Date").alias("date_min"),
            F.max("Date").alias("date_max"),
        )
        .orderBy("ingest_date", "_source_type")
)